# Attention Mechanism

Based on Chapter 3 of *<a href="https://github.com/rasbt/LLMs-from-scratch">Build a Large Language Model from Scratch</a>* by Sebastian Raschka

* **Attention** is a mechanism that helps a model decide which parts of the input are most important when making a prediction.
* **Self-attention** is a special type of attention where the model looks at different parts of the same input to understand relationships within it.

Basically, self-attention is a type of attention, the difference is where this information comes from.

#### **Attention**

English sentence: `The cat sat on the mat.`<br>
Italian sentence: `Il gatto è seduto sul tappetino.`

Therefore, when the model generates word "gatto", it looks at ("attends to") the english word "cat".<br>

In this case:<br>
    * Query: Italian word<br>
    * Key/Value: English sentence

The model is attending to **another sequence**.


#### **Self-attention** 

The model processes a sentence like: `The cat is sleeping, beacuse it is tired.`

When processing the word `it`, the model compares it with every word in the sentence (including the word `it`):
* the<br>
* cat<br>
* is<br>
* sleeping<br>
* ... <br>
and decides that `cat` is the most important word to `it` in the sequence.<br>

In this case: <br>
    * Query: word in the sentence<br>
    * Key/Value: words in the same sentence<br>

The model is attending to **itself**.


## Simplified self-attention with no trainable weights

The goal of this section is to understand the basic idea of attention: first we compute scores, then turn them into weights, and finally use those weights to combine information. Later, we’ll add learnable projections, but for now we focus only on this simple three-step process.

Suppose we are given an input sequence $x^{(1)}$ to $x^{(T)}$. <br>
The input is a text (we will take the example from the textbook: "Your journey starts with one step") that has already been converted into token embeddings as described in chapter 2. <br>
Therefore: 
>$x^{(1)}$ is a d-dimensional vector representing the word "Your"<br>
>$x^{(2)}$ is a vector representing the word "journey"<br>
>$x^{(3)}$ is a vector representing the word "starts", and so forth.
           

In [3]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

print(inputs.shape)

torch.Size([6, 3])


Our **goal** is to compute a context vector ($z^{(i)}$) for each element in the input sequence, mapping each input position from `1` to `T` to a context vector of the *same dimension* as the input.

We'll compute the context vector for a single word first - **"journey"** (index 1 in the input sentence), and then generalize to the whole sentence at once.

### Step 1: Attention scores

By convention, the raw (unnormalized) values are called **attention scores**, and after normalizing them so they sum to 1, they are called **attention weights**.

Since we are computing the context vector for a word **"journey"** our query is simply the token vector ($q^{(2)}$ = $x^{(2)}$)

Suppose we use the second query vector \( $q^{(2)}$ \). Then the attention scores are computed via dot products:

$$
\omega_{21} = x^{(1)} (q^{(2)})^\top
$$

$$
\omega_{22} = x^{(2)} (q^{(2)})^\top
$$

$$
\omega_{23} = x^{(3)} (q^{(2)})^\top
$$

$$
\cdots
$$

$$
\omega_{2T} = x^{(T)} (q^{(2)})^\top
$$
The dot product is a natural similarity measure: vectors pointing in similar directions produce a large dot product, dissimilar vectors produce something smaller.

In [6]:
query = inputs[1]  # "journey"

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Step 2: Normalize scores into attention *weights*

In [7]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In practice, the softmax function for normalization is more robust to very large or very small values and provides better gradient behavior during training, which helps the model learn more effectively.

In [8]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Naive softmax:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Naive softmax: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


But life has been made even easier, because PyTorch provides a built-in softmax function!

In [10]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("torch.softmax:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

torch.softmax: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


### Step 3: Compute the context vector

In [12]:
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### Generalizing to all inputs at once

In [13]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Since for-loops are computationally inefficient, the same result can be obtained more efficiently using matrix multiplication:

In [14]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [17]:
attn_weights = torch.softmax(attn_scores, dim=-1) # apply softmax across each row independently
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [18]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


And the final context vectors for the whole sentence, again in a single matrix multiply:

In [19]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


However, during this process, no actual learning of context takes place, since there are no parameters to update. What we want instead is to let the model learn three separate projections of each word. Q, K, V, finally!

## Self-attention with trainable weights (Q, K, V)

Also known as *scaled dot-product attention*. This is the version used in actual transformers (GPT, BERT, and the one in *Attention Is All You Need*). The three steps `scores -> normalize -> combine` remain exactly the same. The only change is that, before computing the scores, each input is first passed through three separate learned linear projections.

### Three separate projections:

- **Query** - a representation of "what this token is looking for" from other tokens.
- **Key** - a representation of "what this token has to offer," used only for *being matched against* a query.
- **Value** - a representation of "what this token actually contributes" once it's selected by attention.

### Set up the weight matrices

We start by introducing the three training weight matrices $W_q$ , $W_k$ , $W_v$ .<br>

These three matrices are used to project the input embeddings ($x^{(i)}$) into query, key, and value vectors through matrix multiplication:

$$
q^{(i)} = x^{(i)} W_q
$$

$$
k^{(i)} = x^{(i)} W_k
$$

$$
v^{(i)} = x^{(i)} W_v
$$

The embedding dimensions of the input and the query vector can be the same or different, depending on the model design and implementation choices.

In [22]:
x_2 = inputs[1]  # "journey"
d_in = inputs.shape[1]   # 3, the input embedding dim
d_out = 2                   # the output embedding dim

In [30]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [31]:
print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


### Project the input into query, key, value

In [32]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


In [34]:
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


### Attention scores

Using dot product again, but now between the projected query and projected keys, not the raw embeddings.

In [38]:
print(keys_2)

tensor([0.4433, 1.1419])


Let's compute the attention score for $\omega_{22}$:

In [40]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [41]:
# generalize: journey's query against every key at once
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


### Scaling and Softmax

As the key dimension $d_k$ increases, dot products grow in magnitude, which can cause the softmax to become overly sharp and lead to very small gradients. Scaling by `sqrt(d_k)` keeps the variance of the attention scores stable.


In [45]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)
print(attn_weights_2.sum())

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor(1.)


### Weighted sum of values

In [47]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


` project -> scores -> scale -> softmax -> weighted sum of values `

### Reusable class

In [55]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query
        keys    = x @ self.W_key
        values  = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [56]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [57]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [58]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Causal self-attention mechanism

In next-token prediction tasks, allowing a token to attend to future tokens would reveal information that should not yet be available. Therefore, GPT-style models use causal (autoregressive) attention, where each token can attend only to itself and preceding tokens. This is implemented by masking attention scores for all future positions before applying the softmax.

### Step 1: Calculate attention weights (non-causal)

In [59]:
# Reuse the query and key weight matrices of the
# SelfAttention_v2 object from the previous section for convenience
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### Approach 1: Mask, then normalize again

A simple and intuitive way to enforce causality is to construct a lower-triangular mask filled with ones, where entries marked as 1 indicate positions the model is allowed to attend to, and zeros block attention to future tokens. This mask is then applied element-wise to the attention weights, effectively removing any contributions from disallowed (future) positions.
Because masking changes the row sums (the softmax no longer sums to 1 after zeroing entries), each row must be renormalized so that the remaining weights again form a valid probability distribution.

In practice, this can be implemented using `torch.tril`, which conveniently generates this lower-triangular structure.

In [60]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [62]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [63]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


### Approach 2: Mask with `-inf` before the softmax

In [65]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print(mask)  # 1 = future position (forbidden), 0 = allowed

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [66]:
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [67]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


## Masking additional attention weights with dropout

Dropout is a regularization technique used to reduce overfitting by randomly deactivating a fraction of values during training. In practice, each value is independently set to zero with probability p, which forces the model to avoid depending too heavily on any single feature or connection.

In [70]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
example = torch.ones(6, 6)
print(example)
print(dropout(example))

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


PyTorch automatically multiplies surviving values by `1 / (1 - p)` (here, `1 / 0.5 = 2`). This keeps the expected sum of the row roughly constant whether dropout is on or off, so the rest of the network doesn't get a systematically different scale of input during training vs. inference.

In [71]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


## Compact causal self-attention class

Real training happens on batches of sequences, not one sequence at a time.

In [72]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)  # (batch_size=2, num_tokens=6, d_in=3)

torch.Size([2, 6, 3])


In [73]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape # transpose dimnsions 1 and 2
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)

        # batched matmul: (b, num_tokens, d_out) @ (b, d_out, num_tokens)
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [74]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in=3, d_out=2, context_length=context_length, dropout=0.0)
context_vecs = ca(batch)
print(context_vecs.shape)
print(context_vecs)

torch.Size([2, 6, 2])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


## Mutli-head attention

Everything described so far corresponds to a single attention head: one set of query, key, and value projections that produces a single attention pattern and a corresponding set of context vectors. Multi-head attention extends this idea by running several attention heads in parallel. Each head has its own independently learned projection matrices ($W_q$ , $W_k$ , $W_v$), allowing it to focus on different types of relationships in the sequence. The outputs of all heads are then concatenated and combined, enabling the model to integrate multiple representation subspaces and capture a richer set of dependencies.

### So are all the heads looking at the same thing?

No. Each head starts with the same input embeddings, but because each head has separate, independently initialized and independently trained weight matrices, each head is free to learn a different *notion of relevance.*

This is similar to a convolutional neural network using multiple filters/channels in one layer: each filter can learn to detect a different feature (edges, textures, colors).

### Two ways of implementing multi-head attention:

1. **Wrapper approach** - create several independent `CausalAttention` modules (each with its own full-size Q/K/V projections) and concatenate their outputs. This version is conceptually simple and closely matches the idea of “multiple heads”, but it is computationally inefficient because it duplicates the same linear projections across heads.
2. **Weight-splitting (single projection) approach** -  instead of separate modules, use a single larger `nn.Linear` layer for queries (and similarly for keys and values). The output is then reshaped and partitioned into multiple heads along the feature dimension.

## The Wrapper

In [77]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [78]:
torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


## The weight splits approach

In [80]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [81]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
